# cerberus-neuro — v0 Phase 0.5: cell-type single-task deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/02_cell_type_deploy.ipynb)

Train `CellTypeOnlyModel` (encoder + 4-class classifier head) on the v0 16k-crop subset and push the best-epoch checkpoint to Hugging Face Hub as `patrickjreed/cerberus-neuro-cell-type-v0`.

Validated in `02_sanity_check.ipynb` § 3 at val acc 0.96 in 1 epoch with the same data and recipe. This notebook re-runs that training over a few epochs to pick the best checkpoint, then packages it as a shippable HF artifact with a model card.

Runtime: ~15-20 min on Colab Pro L4 (5 epochs × ~200 steps × ~250ms + per-epoch val + HF push).

Per the v0 phase plan ([spec](../docs/superpowers/specs/2026-05-08-v0-baseline-first-paired-experiment-design.md)), this is Phase 0.5 — a parallel deployment task that ships a concrete v0 artifact independent of Phase 2 (multi-task Cerberus) outcome.

## 1. Setup

In [ ]:
# Install the argus_cells package code only (--no-deps) so pip never touches
# Colab's pre-built numpy / scipy / scikit-learn. Upgrading those mid-session
# breaks the numpy<->scipy ABI (the "_blas_supports_fpe" / "_center" ImportErrors).
!pip install -q --upgrade --no-deps git+https://github.com/PatrickJReed/argus-cells.git@main
# Only the pure-Python deps Colab may lack; these pull NO numpy-linked packages.
!pip install -q boto3 huggingface_hub

import sys
for _m in list(sys.modules):
    if _m == 'argus_cells' or _m.startswith('argus_cells.'):
        del sys.modules[_m]
print('install + cache purge done')

In [ ]:
import os
from pathlib import Path
import torch

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF login: OK (Colab secret)')
except Exception as e:
    print(f'HF login skipped ({type(e).__name__}); HF push will be disabled')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_BASE = Path('/content/drive/MyDrive/cerberus-neuro/cache/v0')
    CACHE_DIR = Path('/content/cerberus-cache')
except Exception:
    CKPT_BASE = Path.home() / '.cache' / 'cerberus-neuro' / 'v0'
    CACHE_DIR = CKPT_BASE

CKPT_BASE.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'CACHE_DIR: {CACHE_DIR}')
print(f'CKPT_BASE: {CKPT_BASE}')
print(f'GPU: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Manifest + train/val split + prefetch

In [ ]:
from argus_cells.data import (
    BUCKET, WORKSPACE_PREFIX, CHANNEL_ORDER,
    build_manifest, subset_manifest, well_level_split, NeuroPaintingDataset,
)

manifest = build_manifest(CACHE_DIR)
subset = subset_manifest(manifest, wells_per_cell_type=48, sites_per_well=9, seed=0)
train_manifest, val_manifest = well_level_split(subset, val_frac=0.2, seed=0)
print(f'train: {len(train_manifest)} sites, val: {len(val_manifest)} sites')

# Parallel-prefetch all referenced TIFFs and Cells.csv into /content/.
# Reuses cache from prior 02_train.ipynb session if present (skips will dominate).
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor, as_completed

PREFETCH_WORKERS = 32
s3_client = boto3.client('s3', config=Config(
    signature_version=UNSIGNED,
    max_pool_connections=PREFETCH_WORKERS,
))

def _expected_keys(manifest):
    keys = set()
    for _, row in manifest.iterrows():
        for stain in CHANNEL_ORDER:
            url = str(row[f'URL_{stain}'])
            keys.add(url.replace(f's3://{BUCKET}/', '').lstrip('/'))
        keys.add(
            f"{WORKSPACE_PREFIX}analysis/{row['batch']}/{row['Metadata_Plate']}/"
            f"analysis/{row['Metadata_Plate']}-{row['Metadata_Well']}-{row['Metadata_Site']}/Cells.csv"
        )
    return sorted(keys)

def _download_one(key):
    local = CACHE_DIR / key
    if local.exists():
        return 'skip'
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        s3_client.download_file(BUCKET, key, str(local))
        return 'ok'
    except Exception as e:
        return f'fail: {e}'

import time
all_keys = _expected_keys(subset)
print(f'prefetch target: {len(all_keys):,} files')
t0 = time.time()
n_ok = n_skip = n_fail = 0
with ThreadPoolExecutor(max_workers=PREFETCH_WORKERS) as exe:
    futures = {exe.submit(_download_one, k): k for k in all_keys}
    for i, fut in enumerate(as_completed(futures), 1):
        result = fut.result()
        if result == 'ok': n_ok += 1
        elif result == 'skip': n_skip += 1
        else: n_fail += 1
        if i % 1000 == 0 or i == len(all_keys):
            print(f'  {i:>6}/{len(all_keys):>6}  ok={n_ok}  skip={n_skip}  fail={n_fail}')
print(f'prefetch done: {n_ok} downloaded, {n_skip} cached, {n_fail} failed in {time.time()-t0:.0f}s')

## 3. Train cell-type classifier

Same training recipe that produced 0.96 val acc in 1 epoch in the sanity check. 5 epochs at cosine-annealed LR; save best-epoch checkpoint.

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from argus_cells.model import CellTypeOnlyModel

torch.set_float32_matmul_precision('high')
torch.manual_seed(0)

BATCH_SIZE = 64
NUM_WORKERS = 8
CROPS_PER_SITE = 12
N_EPOCHS = 5

train_ds = NeuroPaintingDataset(
    train_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=1, augment=True,
)
val_ds = NeuroPaintingDataset(
    val_manifest, CACHE_DIR,
    crop_size=256, crops_per_site=CROPS_PER_SITE,
    min_cells_per_crop=1, augment=False, shuffle=False,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          persistent_workers=True, pin_memory=True, prefetch_factor=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          persistent_workers=True, pin_memory=True, prefetch_factor=4)

model = CellTypeOnlyModel().to(device)
encoder_params = list(model.encoder.parameters())
head_params = list(model.head.parameters())
optimizer = AdamW([
    {'params': encoder_params, 'lr': 3e-5},
    {'params': head_params, 'lr': 3e-4},
], weight_decay=1e-4)

steps_per_epoch = max(1, (len(train_manifest) * CROPS_PER_SITE) // BATCH_SIZE)
total_steps = N_EPOCHS * steps_per_epoch
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

print(f'steps_per_epoch={steps_per_epoch}  total_steps={total_steps}')

ct_dir = CKPT_BASE / 'cell_type'
ct_dir.mkdir(parents=True, exist_ok=True)

step = 0
best_val_acc = 0.0
history = []
for epoch in range(N_EPOCHS):
    model.train()
    train_loss = train_correct = train_n = 0
    for bf, fluo, ct, cond in train_loader:
        bf, ct = bf.to(device, non_blocking=True), ct.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        ctx = torch.amp.autocast('cuda') if scaler is not None else torch.amp.autocast('cuda', enabled=False)
        with ctx:
            logits = model(bf)
            loss = F.cross_entropy(logits, ct)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        train_loss += loss.item() * bf.size(0)
        train_correct += (logits.argmax(1) == ct).sum().item()
        train_n += bf.size(0)
        step += 1
        if step % 25 == 0:
            print(f'  step {step:4d}: loss={loss.item():.4f}')
        if step >= (epoch + 1) * steps_per_epoch:
            break

    model.eval()
    val_correct = val_n = 0
    val_loss_sum = 0.0
    with torch.no_grad():
        for bf, fluo, ct, cond in val_loader:
            bf, ct = bf.to(device), ct.to(device)
            logits = model(bf)
            val_loss_sum += F.cross_entropy(logits, ct).item() * bf.size(0)
            val_correct += (logits.argmax(1) == ct).sum().item()
            val_n += bf.size(0)
    val_acc = val_correct / val_n
    val_loss = val_loss_sum / val_n
    history.append({'epoch': epoch, 'train_loss': train_loss/train_n, 'train_acc': train_correct/train_n,
                    'val_loss': val_loss, 'val_acc': val_acc})
    print(f'epoch {epoch}: train_loss={train_loss/train_n:.4f}  '
          f'train_acc={train_correct/train_n:.4f}  '
          f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'val_acc': val_acc,
            'val_loss': val_loss,
            'history': history,
            'config': {
                'in_channels': 1, 'n_classes': 4,
                'pretrained_encoder': True,
                'batch_size': BATCH_SIZE, 'lr_encoder': 3e-5, 'lr_head': 3e-4,
                'n_epochs': N_EPOCHS, 'crops_per_site': CROPS_PER_SITE,
                'wells_per_cell_type': 48, 'sites_per_well': 9,
            },
        }, ct_dir / 'best.pt')
        print(f'  -> new best at epoch {epoch}, val_acc={val_acc:.4f}; saved to {ct_dir}/best.pt')

print(f'\nbest val_acc = {best_val_acc:.4f} at epoch {best_epoch}')
print(f'training history: {history}')

## 4. Push to Hugging Face Hub

Upload the best-epoch checkpoint to `patrickjreed/cerberus-neuro-cell-type-v0` along with a model card.

In [ ]:
from huggingface_hub import HfApi, create_repo

REPO_ID = 'patrickjreed/cerberus-neuro-cell-type-v0'

create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)
api = HfApi()

# Upload the best checkpoint as best.pt (durable filename, not epoch_NNN).
api.upload_file(
    path_or_fileobj=str(ct_dir / 'best.pt'),
    path_in_repo='best.pt',
    repo_id=REPO_ID, repo_type='model',
)
print(f'uploaded best.pt to {REPO_ID}')

# Write the model card.
history_table = '\n'.join(
    f"| {h['epoch']} | {h['train_loss']:.4f} | {h['train_acc']:.4f} | {h['val_loss']:.4f} | {h['val_acc']:.4f} |"
    for h in history
)
model_card = f'''---
license: mit
library_name: pytorch
tags:
- biology
- cell-painting
- microscopy
- image-classification
- neuropainting
datasets:
- cellpainting-gallery/cpg0038-tegtmeyer-neuropainting
metrics:
- accuracy
---

# cerberus-neuro: cell-type classifier (v0)

Brightfield-only 4-class cell-type classifier on the Broad NeuroPainting Cell Painting dataset. Part of the [cerberus-neuro project](https://github.com/PatrickJReed/cerberus-neuro), which trains task-specific models on a shared ResNet34 architecture for cell-type classification, organelle soft-segmentation, and disease-state classification.

This is the **cell-type single-task model**, shipped as Phase 0.5 of the v0 plan. The encoder is initialized from ImageNet1K_V1 weights with `conv1` adapted to 1-channel brightfield input via mean-across-RGB-channels. Trained end-to-end with discriminative learning rates.

## Inputs

- 1-channel brightfield image, 256x256, 16-bit normalized to [0, 1]

## Outputs

4-class softmax over `{{0: stem (iPSC), 1: progen (NPC), 2: neuron, 3: astro (astrocyte)}}`.

## Validation results

Best-epoch validation accuracy: **{best_val_acc:.4f}** (epoch {best_epoch}). Trained for {N_EPOCHS} epochs on the v0 16k-crop subset (48 wells per cell type, 9 sites per well, top-12 cell-rich tiles per site, well-level 80/20 train/val split).

| epoch | train_loss | train_acc | val_loss | val_acc |
|---|---|---|---|---|
{history_table}

## Training recipe

- ResNet34 encoder, ImageNet1K_V1 pretrained, conv1 adapted for 1-channel input via mean-across-RGB.
- ClassifierHead: AdaptiveAvgPool2d -> Linear(512, 4).
- AdamW, encoder LR `3e-5`, head LR `3e-4` (discriminative, 0.1x ratio).
- Cosine annealing over total_steps; AMP autocast + GradScaler; gradient norm clip at 1.0.
- Batch size 64, num_workers 8, prefetch_factor 4.
- D4 dihedral augmentation (rotation x flip) applied identically across all 6 channels per crop.
- Cell-aware tile selection: top-12 256x256 patches per site by CellProfiler centroid count.

## Loading

```python
import torch
from huggingface_hub import hf_hub_download
from argus_cells.model import CellTypeOnlyModel

ckpt_path = hf_hub_download('{REPO_ID}', 'best.pt')
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

model = CellTypeOnlyModel(in_channels=1, n_classes=4, pretrained_encoder=False)
model.load_state_dict(ckpt['model'])
model.eval()
```

## Limitations

- Trained on Broad NeuroPainting only (cpg0038); may not transfer to other Cell Painting protocols without fine-tuning.
- Trained on 20x acquisition; behavior on 63x is not characterized.
- Data subset is 48 wells per cell type — strong validation but not exhaustive across the full cohort.
- 4-way classification only; does not predict disease state or fluorescence.

## Citation and related work

- Tegtmeyer et al., *Nat Commun* (2025), [10.1038/s41467-025-61547-x](https://doi.org/10.1038/s41467-025-61547-x). The NeuroPainting dataset.
- Lyu, Alonso, Pinto. *Cerberus: A Multi-headed Network for Brain Tumor Segmentation*, BrainLes 2020. The shared-encoder + multi-head architectural pattern this model is part of.

## License

MIT.
'''

(ct_dir / 'README.md').write_text(model_card)
api.upload_file(
    path_or_fileobj=str(ct_dir / 'README.md'),
    path_in_repo='README.md',
    repo_id=REPO_ID, repo_type='model',
)
print(f'uploaded README.md (model card) to {REPO_ID}')

print(f'\ndone. View at: https://huggingface.co/{REPO_ID}')